## Downaload a dataset

In [1]:
from huggingface_hub import login
login()

In [2]:
from datasets import load_dataset

# Load a specific language subset (recommended — full dataset is huge)
ds = load_dataset("HuggingFaceFW/fineweb-2", name="ita_Latn", split="train", streaming=True)

Resolving data files:   0%|          | 0/85 [00:00<?, ?it/s]

### Load tokenizer

In [3]:
import sys
sys.path.append('..')

In [4]:
from minbpe import RegexTokenizer
import numpy as np

tokenizer = RegexTokenizer()
tokenizer.load(model_file="../output/tokenizer/fineweb_tokenizer.model")
EOT_TOKEN = tokenizer.special_tokens["<|endoftext|>"]

max_token = len(tokenizer.vocab) + len(tokenizer.special_tokens)
DTYPE = np.uint16 if max_token < (1 << 16) else np.uint32

## Stream the dataset & apply length filter

By Chinchilla's law (20 tokens/param), a ~125M params model needs ~2.5B tokens. At ~4.5 characters per token, that's ~11.25 GB of raw text — I collect 12 GB to leave a small buffer for filtering losses during tokenization.

In [5]:
from multiprocessing import cpu_count

MIN_CHARS      = 100
MAX_CHARS      = 100_000
TARGET_BYTES   = 12 * (1024 ** 3)                # ~12 GB of raw text
VAL_FRACTION   = 0.0005
SEED           = 3647
NUM_PROC       = cpu_count() // 2                # workers for .map()

In [6]:
from tqdm import tqdm

texts = []
total_bytes = 0

pbar = tqdm(total=TARGET_BYTES, desc="Collecting text", unit="B", unit_scale=True)
for sample in ds:
    t = sample["text"]
    n = len(t)
    if n < MIN_CHARS or n > MAX_CHARS:
        continue
    texts.append(t)
    total_bytes += n
    pbar.update(n)
    if total_bytes >= TARGET_BYTES:
        break

pbar.close()
print(f"Collected {len(texts):,} documents  ({total_bytes / 1e9:.2f} GB)")

Collected 4,158,371 documents  (12.88 GB)


## Build a HF Dataset from the collected texts

In [7]:
from datasets import Dataset, DatasetDict

full_ds = Dataset.from_dict({"text": texts})
del texts  # free memory

split_dataset = full_ds.train_test_split(
    test_size=VAL_FRACTION, seed=SEED, shuffle=True
)
split_dataset["val"] = split_dataset.pop("test")
print(split_dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 4156291
    })
    val: Dataset({
        features: ['text'],
        num_rows: 2080
    })
})


## Tokenize

In [ ]:
def process(example):
    # encode_ordinary ignores special tokens; single-process per worker
    ids = tokenizer.encode_ordinary(example["text"], n_workers=1)
    ids.append(EOT_TOKEN)
    return {"ids": ids, "len": len(ids)}

tokenized = split_dataset.map(
    process,
    remove_columns=["text"],
    desc="Tokenizing",
    num_proc=cpu_count(),
)

del split_dataset  # free memory

In [3]:
lengths = [example["len"] for example in tokenized]

print(f"Total examples: {len(lengths)}")
print(f"Total tokens:   {sum(lengths)}")
print(f"Mean length:    {np.mean(lengths):.1f}")
print(f"Median length:  {np.median(lengths):.1f}")
print(f"Std dev:        {np.std(lengths):.1f}")
print(f"Min length:     {np.min(lengths)}")
print(f"Max length:     {np.max(lengths)}")
print(f"25th pct:       {np.percentile(lengths, 25):.1f}")
print(f"75th pct:       {np.percentile(lengths, 75):.1f}")
print(f"90th pct:       {np.percentile(lengths, 90):.1f}")
print(f"99th pct:       {np.percentile(lengths, 99):.1f}")

Total examples:      4,158,371
Total tokens:    2,687,428,344
Mean length:             646.3
Median length:           389.0
Std dev:                 981.8
Min length:                 53
Max length:             52,692
25th pct:                203.0
75th pct:                721.0
90th pct:               1314.0
99th pct:               4542.0


## Save the tokenized dataset

In [ ]:
import os
import math

BATCH_SIZE = 10_000

for split, dset in tokenized.items():
    arr_len = np.sum(dset["len"], dtype=np.uint64)
    filename = os.path.join("../output/encoded_data/", f"{split}.bin")
    print(f"{split}: {arr_len:,} tokens  ->  {filename}")
    arr = np.memmap(filename, dtype=DTYPE, mode="w+", shape=(arr_len,))

    num_batches = math.ceil(len(dset) / BATCH_SIZE) if len(dset) > BATCH_SIZE else 1
    idx = 0
    for batch_idx in tqdm(range(num_batches), desc=f"Writing {split}.bin"):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, len(dset))
        batch = dset.select(range(start, end)).with_format("numpy")
        arr_batch = np.concatenate(batch["ids"])
        arr[idx : idx + len(arr_batch)] = arr_batch
        idx += len(arr_batch)
    arr.flush()

print("Done.")